In [ ]:
import pandas as pd
from difflib import get_close_matches

df = pd.read_csv(
    r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new(target portfolio ).csv',
    encoding='latin-1'
)
print(df.columns)

df['company_normalized'] = df['company '].str.strip().str.lower()

unique_companies = df['company_normalized'].unique()
name_mapping = {}

for company in unique_companies:
    matches = get_close_matches(company, unique_companies, n=5, cutoff=0.85)
    if len(matches) > 1:
        canonical = min(matches, key=len)
        for match in matches:
            name_mapping[match] = canonical
    else:
        name_mapping[company] = company

df['company_canonical'] = df['company_normalized'].map(name_mapping)

years = sorted(df['year'].unique())
pivot_data = []

for company in df['company_canonical'].unique():
    company_data = df[df['company_canonical'] == company]
    row = {'company': company}
    
    for year in years:
        year_data = company_data[company_data['year'] == year]
        if not year_data.empty:
            row[f'region_{year}'] = year_data['region'].iloc[0]
            row[f'sector_{year}'] = year_data['sector '].iloc[0]
            row[f're100_{year}'] = year_data['re100'].iloc[0]
            row[f'sbti_{year}'] = year_data['sbti'].iloc[0]
            row[f'cn_{year}'] = year_data['cn'].iloc[0]
            row[f'nz_{year}'] = year_data['nz'].iloc[0]
            row[f'cc_{year}'] = year_data['cc'].iloc[0]
    
    pivot_data.append(row)

matrix = pd.DataFrame(pivot_data)
matrix.to_csv('company_matrix.csv', index=False)

print(unique_companies)



Index(['year', 'company ', 'region', 'sector ', 're100', 'sbti', 'cn', 'nz',
       'cc', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11'],
      dtype='object')
0                             3m
1                            abb
2            abbott laboratories
3                         abbvie
4                      accenture
                  ...           
2495                     wistron
2496                   medtronic
2497        china life insurance
2498    china general technology
2499                    heineken
Name: company_canonical, Length: 2500, dtype: object


In [8]:
print(unique_companies)
print(len(unique_companies))


['3m' 'abb' 'abbott laboratories' 'abbvie' 'accenture' 'achmea' 'acs'
 'aegon' 'aeon' 'agricultural bank of china' 'aia group' 'airbus' 'aisin'
 'albertsons' 'alfresa holdings' 'alibaba group holding'
 'alimentation couche-tard' 'allianz' 'allstate' 'alphabet'
 'aluminum corp. of china' 'amazon' 'amer international group'
 'américa móvil' 'american express' 'american international group'
 'amerisourcebergen' 'amgen' 'anglo american' 'anheuser-busch inbev'
 'anhui conch group' 'ansteel group' 'anthem' 'apple' 'arcelormittal'
 'archer daniels midland' 'arrow electronics' 'assicurazioni generali'
 'astrazeneca' 'at&t' 'aviation industry corp. of china' 'aviva' 'axa'
 'bae systems' 'banco bilbao vizcaya argentaria' 'banco bradesco'
 'banco do brasil' 'banco santander' 'bank of america' 'bank of china'
 'bank of communications' 'bank of montreal' 'bank of nova scotia'
 'barclays' 'basf' 'bayer' 'beijing automotive group'
 'beijing jianlong heavy industry group' 'berkshire hathaway' 'best bu

In [9]:
print(company_data)

      year                  company          region  \
2489  2025  GuideWell Mutual Holding  North America   

                              sector   re100  sbti  cn  nz  cc  Unnamed: 9  \
2489  Health care and pharmaceuticals    NaN   NaN NaN NaN NaN         NaN   

      Unnamed: 10  Unnamed: 11        company_normalized  \
2489          NaN          NaN  guidewell mutual holding   

             company_canonical  
2489  guidewell mutual holding  


In [3]:
import pandas as pd
from difflib import get_close_matches

df = pd.read_csv(
    r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new(target portfolio ).csv',
    encoding='latin-1'
)

df['company_normalized'] = df['company '].str.strip().str.lower()

unique_companies = df['company_normalized'].unique()
name_mapping = {}

for company in unique_companies:
    matches = get_close_matches(company, unique_companies, n=5, cutoff=0.85)
    if len(matches) > 1:
        canonical = min(matches, key=len)
        for match in matches:
            name_mapping[match] = canonical
    else:
        name_mapping[company] = company

df['company_canonical'] = df['company_normalized'].map(name_mapping)

years = sorted(df['year'].unique())
targets = ['re100', 'sbti', 'cn', 'nz', 'cc']
results = []

for company in df['company_canonical'].unique():
    company_data = df[df['company_canonical'] == company]
    row = {'company': company}
    
    for target in targets:
        transition = []
        for year in years:
            year_data = company_data[company_data['year'] == year]
            if year_data.empty:
                transition.append('-')
            else:
                val = year_data[target].iloc[0]
                if pd.isna(val):
                    transition.append('0')
                elif val == 1:
                    transition.append('1')
                elif val == -1:
                    transition.append('-1')
                else:
                    transition.append('0')
        row[target] = ''.join(transition)
    
    results.append(row)

output = pd.DataFrame(results)
output.to_csv('company_transitions_map_cc_fixed.csv', index=False)



# SBTI to net zero

In [9]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')
lost_sbti = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    if '10' in sbti:
        lost_sbti.append(row['company'])

print(f"Companies that lost SBTi: {len(lost_sbti)}")
print(lost_sbti)

Companies that lost SBTi: 15
['abbott laboratories', 'bouygues', 'ck hutchison holdings', 'coop group', 'deutsche bank', 'elo group', 'ing group', 'metro', 'mitsui', 'nippon telegraph and telephone', 'sncf group', 'tata motors', 'verizon communications', 'america movil', 'novo nordisk']


In [10]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

lost_sbti = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    if '10' in sbti:
        sbti_loss_pos = sbti.index('10')
        nz_after = row['nz'].replace('-', '')[sbti_loss_pos+1:]
        cn_after = row['cn'].replace('-', '')[sbti_loss_pos+1:]
        
        if '1' in nz_after or '1' in cn_after:
            lost_sbti.append({
                'company': row['company'],
                'sbti': row['sbti'],
                'nz': row['nz'],
                'cn': row['cn']
            })

pd.DataFrame(lost_sbti).to_csv('sbti_lost_then_gained.csv', index=False)

lost_sbti_2023_2025 = []
for _, row in df.iterrows():
    sbti = row['sbti'][2:5].replace('-', '')
    nz = row['nz'][2:5].replace('-', '')
    
    if '10' in sbti:
        lost_pos = sbti.index('10')
        nz_after = nz[lost_pos+1:]
        gained_nz = '1' in nz_after
        lost_sbti_2023_2025.append({'company': row['company'], 'gained_nz': gained_nz})

result = pd.DataFrame(lost_sbti_2023_2025)
print(f"Lost SBTi 2023-2025: {len(result)}")
print(f"Gained NZ: {result['gained_nz'].sum()}")

Lost SBTi 2023-2025: 12
Gained NZ: 3


## fixed 


In [12]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

# Total lost SBTi
lost_sbti = 0
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            lost_sbti += 1
            break

print(f"Lost SBTi: {lost_sbti}")

# Lost SBTi and gained NZ or CN
lost_sbti_gained = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    nz = row['nz'].replace('-', '')
    cn = row['cn'].replace('-', '')
    
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            if '1' in nz[i+1:] or '1' in cn[i+1:]:
                lost_sbti_gained.append({
                    'company': row['company'],
                    'sbti': row['sbti'],
                    'nz': row['nz'],
                    'cn': row['cn']
                })
            break

result = pd.DataFrame(lost_sbti_gained)
result.to_csv('sbti_lost_then_gained.csv', index=False)
print(f"Lost SBTi and gained NZ/CN: {len(result)}")

print(lost_sbti)

Lost SBTi: 15
Lost SBTi and gained NZ/CN: 10
15


# carbon neutral to net zero

In [8]:

import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

lost_cn_gained_nz = []
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    nz = row['nz'].replace('-', '')
    
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            if '1' in nz[i+1:]:
                lost_cn_gained_nz.append({'company': row['company'], 'cn': row['cn'], 'nz': row['nz']})
            break

result = pd.DataFrame(lost_cn_gained_nz)
result.to_csv('lost_cn_gained_nz.csv', index=False)
print(result)

lost_cn = 0
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            lost_cn += 1
            break

print(f"Lost CN: {lost_cn}")

                             company     cn     nz
0                                abb  11-00  00-10
1                               aeon  10000  01110
2              alibaba group holding  01110  00001
3           alimentation couche-tard  00010  00001
4                            allianz  11000  11110
..                               ...    ...    ...
159  contemporary amperex technology  --010  --001
160                  lufthansa group  --110  --001
161                    tongwei group  --110  --001
162      luxshare precision industry  --110  --001
163                 societe generale  ---10  ---01

[164 rows x 3 columns]
Lost CN: 199


In [ ]:
# Track CN 2021 -> NZ 2025
cn_to_nz = matrix[
    (matrix['cn_2021'] == 1) & 
    (matrix['nz_2025'] == 1)
]['company'].tolist()

# Track SBTi changes every two years
sbti_transitions = {}
for i in range(len(years) - 1):
    year1, year2 = years[i], years[i+1]
    if year2 - year1 <= 2:
        gained = matrix[
            (matrix[f'sbti_{year1}'] != 1) & 
            (matrix[f'sbti_{year2}'] == 1)
        ]['company'].tolist()
        lost = matrix[
            (matrix[f'sbti_{year1}'] == 1) & 
            (matrix[f'sbti_{year2}'] != 1)
        ]['company'].tolist()
        sbti_transitions[f'{year1}_to_{year2}'] = {'gained': gained, 'lost': lost}

print(f"Companies with CN in 2021 and NZ in 2025: {len(cn_to_nz)}")
print(f"\nSBTi transitions: {sbti_transitions}")

matrix.to_csv('company_matrix.csv', index=False)

that brings the 